# UI Explore Playground

Подбор локаторов для autoUI: императивный обход pywinauto и проверка `Locator` pipeline.

Требования: Windows, открытое целевое приложение (по умолчанию Notepad++).

In [ ]:
import os

os.chdir("..")
os.getcwd()

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from autoui.explore import (
    ExplorerSession,
    find_desktop_window,
    list_menu_items,
    path_from_root,
)
from autoui.locators import ChildOp, FindDescendantsOp, Locator, TakeOp

In [ ]:
WINDOW_TITLE = "Notepad++"

session = ExplorerSession(WINDOW_TITLE, verbose_locators=True)
window = session.connect(index=0)
window

## Императивный поиск

Меняйте индексы и фильтры; `session.list_indexed(...)` помогает подобрать номера.

In [ ]:
# --- меняйте запрос ---
WINDOW_INDEX = 0
CHILD_INDEX = 6
DESC_FILTERS = {"control_type": "Button"}
CANDIDATE_INDEX = 22

windows = find_desktop_window(WINDOW_TITLE)
window = windows[WINDOW_INDEX]
areas = window.children()
toolbar = areas[CHILD_INDEX]
candidates = toolbar.descendants(**DESC_FILTERS)
target = candidates[CANDIDATE_INDEX]

session.highlight(target, colour="green")
print(path_from_root(target))

In [ ]:
# Список детей / кандидатов с индексами
session.list_indexed(areas)
# session.list_indexed(candidates)

## Locator pipeline

Проверка через `PywinautoDriver.resolve`; при ошибке — trace в stdout.

In [ ]:
LOCATOR = Locator([
    ChildOp(CHILD_INDEX),
    FindDescendantsOp(where=DESC_FILTERS),
    TakeOp(CANDIDATE_INDEX),
])

element = session.try_locator(LOCATOR)
element

In [ ]:
# Shorthand: Locator.find(control_type="MenuBar")
menu_locator = Locator([
    FindDescendantsOp(where={"control_type": "MenuBar"}),
    TakeOp(0),
    FindDescendantsOp(where={"name": "Search"}),
    TakeOp(0),
])
session.try_locator(menu_locator)

## Меню (опционально)

In [ ]:
list_menu_items(session.window.menu())

In [ ]:
session.clear_highlight()